# Reproducing Kim et al. (2022) from dandisets 000489 and 000448

**Paper.** Kim V, Semenov I, Kiester AS, Keppler MA, Ibey BL, Bixler JN, Pakhomov AG (2022).
Action spectra and mechanisms of (in)efficiency of bipolar electric pulses at electroporation.
*Bioelectrochemistry* 149:108319. [doi:10.1016/j.bioelechem.2022.108319](https://doi.org/10.1016/j.bioelechem.2022.108319).
The claims file (`claims/kim-2022.yaml`) was written from the accepted author manuscript in
PubMed Central (PMC9729435); the paper has no supplement and links no code.

**Dandisets.**
[DANDI:000489](https://dandiarchive.org/dandiset/000489/0.230518.1811) version 0.230518.1811 and
[DANDI:000448](https://dandiarchive.org/dandiset/000448/0.230421.1844) version 0.230421.1844.
The paper cites no version of either. Each dandiset has one published version, the one pinned;
the drafts list the same number of assets and bytes (18 files, 18594422294 and 3173065152 bytes)
and were last modified before the publication of these versions, so the published versions are
the most recent and match the claims file's description as well as the drafts do.

**What the dandisets hold.** 000489 holds 17 stitched dish images of the Fig. 2 experiment
(BPAE cells, 5 pulses of 600 ns or 600+600 ns at 1 Hz, second phase 0, 25, 50, 75 or 100%) and one
sample image labelled Fig1F. 000448 holds strobe image stacks of 9 CHO cells (5 unipolar, 4
bipolar; two trials each) given 5 pulses at 833 kHz (Fig. 7); the paper used 18 and 17 cells.
Neither holds the experiments of Figs. 3 to 6 or the thermometry of Fig. 1E. Dandisets 000548,
000549 and 000550 of the same lab name this paper and match Figs. 3, 4 and 6 by title; they are
not among the paper's datasets that this attempt tests (`papers/datasets/kim-2022.yaml`), so the
claims on Figs. 3 to 6 are blocked here as resting on datasets not used.

**Inputs from outside the dandisets.** None. The field strength of each ROI, which the paper took
from a finite element model whose output is not deposited, is computed here from the geometry in
Methods 2.3 (helper `kim2022.field_kv_cm`).

| Claim | Dataset | Tested here |
|---|---|---|
| fig2a-uptake-rises-with-field | 000489 | tested |
| fig2a-vertical-scales | 000489 | tested |
| fig2b-exponential-same-slope | 000489 | tested |
| fig2c-max-inhibition-50-75 | 000489 | tested |
| fig2c-inhibition-flat-above-8-7 | 000489 | tested |
| fig2c-weaker-at-6-7 | 000489 | tested |
| fig7-unipolar-pole-polarization | 000448 | tested |
| fig7-charging-time-constant | 000448 | tested |
| fig7-later-pulses-same-tau-difference | 000448 | tested |
| fig7-bipolar-overshoot | 000448 | tested |
| fig7-bipolar-alternation | 000448 | tested |
| fig7-summation-without-buildup | 000448 | tested |
| fig3-uptake-linear-in-pulse-number, fig3d-cancellation-independent-of-number | 000489 | blocked: dataset missing (likely in 000548, not used) |
| fig4ac-ns-bipolar-20-25pct, fig4d-1200ns-tenfold, fig4eg-us-bipolar-30-50pct, fig4h-us-stronger-at-low-field | 000489 | blocked: dataset missing (likely in 000549, not used) |
| fig5-ns-symmetric-along-aa, fig5-us-anode-stronger | 000489 | blocked: dataset missing (likely in 000549, not used) |
| fig6ad-unipolar-833khz-enhanced, fig6be-bipolar-khz-less-than-1hz, fig6cf-833khz-thirtyfold, fig6cf-cancellation-stronger-at-high-field | 000489 | blocked: dataset missing (likely in 000550, not used) |
| fig1e-heating | 000489 | blocked: data missing (thermometry photographs not deposited) |

The Fig. 7 claims are tested first, as their data load in two minutes; the dish images take
about six.

## Setup

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from scipy import stats
import kim2022 as k

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("versions:", k.VERSIONS)

## Data check: dandiset 000448 (strobe imaging, Fig. 7)

Each file holds one trial of one cell (or small group of cells): 500 frames of
512 x 512 pixels. The session description gives the pulse shape and whether the
delays were stepped forward (F) or in reverse (R). Methods 2.6: two trials per
cell, ascending and descending delays, averaged.

In [ ]:
strobe_assets = k.list_assets("000448")
with ThreadPoolExecutor(3) as ex:
    strobe = list(ex.map(k.process_strobe, strobe_assets))
st_meta = pd.DataFrame([{kk: s[kk] for kk in ["path", "subject_id", "cell", "pulse_shape", "recording", "rate",
                                               "field", "rate_label", "n_pulses", "start", "shape"]} for s in strobe])
print(st_meta.to_string())

In [ ]:
print("files:", len(strobe), "| cells:", st_meta.cell.nunique())
print(st_meta.groupby("pulse_shape").cell.nunique().rename("cells per pulse shape").to_string())
print("frames per file:", sorted(set(s["shape"][0] for s in strobe)), "| frame rate (Hz):", sorted(set(s["rate"] for s in strobe)),
      "-> frame step (ns):", 1e9 / strobe[0]["rate"])
print("trials per cell:", st_meta.groupby("cell").recording.apply(lambda r: "".join(sorted(r))).value_counts().to_dict())
for s in strobe:
    wc = s["frame_mean"]
    dk = k.dark_frames(wc)
    print(f"{s['subject_id']:6s} raw frame mean {wc.min():7.1f} to {wc.max():7.1f} counts; dark frames {np.nonzero(dk)[0].tolist()};"
          f" mean of frames 1-50 {wc[1:51].mean():.0f}, of frames 450-499 {wc[450:].mean():.0f}; membrane/upper/lower px {s['npx_85_45']}")
# duplicates: identical frame-mean series would mean a trial deposited twice
fm = np.array([s["frame_mean"] for s in strobe])
cc = np.corrcoef(fm)
np.fill_diagonal(cc, 0)
print("largest correlation of frame-mean series between two files:", cc.max().round(3),
      "| identical pairs:", [(strobe[i]["subject_id"], strobe[j]["subject_id"]) for i in range(len(strobe))
                             for j in range(i + 1, len(strobe)) if np.array_equal(fm[i], fm[j])])

The frame-averaged signal falls with the frame index in both F and R files, so the
frames are stored in acquisition order: an R file holds the delays in descending
order and is reversed before averaging. The two trials of a cell are then aligned
by the lag that best correlates their upper-minus-lower pole signals (the absolute
delay of the first frame is not in the files), and averaged.

In [ ]:
fig, axs = plt.subplots(3, 6, figsize=(18, 8))
for ax, s in zip(axs.ravel(), strobe):
    memb, up, lo = s["masks"]
    ax.imshow(s["mean_img"], cmap="gray")
    ov = np.zeros(memb.shape + (4,))
    ov[up] = [1, 0, 0, 0.6]
    ov[lo] = [0, 0.4, 1, 0.6]
    ax.imshow(ov)
    ax.set_title(f"{s['subject_id']} {s['pulse_shape'][:3]}", fontsize=8)
    ax.axis("off")
plt.suptitle("Frame-averaged images; upper pole red, lower pole blue (assumed pole regions)")
plt.tight_layout()
plt.show()

In [ ]:
def build_cells(key="traces_85_45"):
    """Per cell: F and R trials, R reversed, dark frames removed, R aligned to F,
    averaged; returns the averaged upper and lower ratio traces and the lag."""
    cells = {}
    for c in sorted(st_meta.cell.unique()):
        tr = {}
        for s in strobe:
            if s["cell"] != c:
                continue
            t = s[key]
            dk = k.dark_frames(t["whole"])
            u = np.where(dk, np.nan, t["upper"]).astype(float)
            l = np.where(dk, np.nan, t["lower"]).astype(float)
            if s["recording"] == "reverse":
                u, l = u[::-1], l[::-1]
            tr[s["recording"]] = (u, l, s["pulse_shape"])
        uf, lf, shape = tr["forward"]
        ur, lr, _ = tr["reverse"]
        lag = k.align_lag(uf - lf, ur - lr)
        ur, lr = k.shift(ur, lag), k.shift(lr, lag)
        cells[c] = {"shape": shape, "lag": lag, "uf": uf, "lf": lf, "ur": ur, "lr": lr,
                    "u": np.nanmean([uf, ur], axis=0), "l": np.nanmean([lf, lr], axis=0)}
    return cells

cells = build_cells()
print("lag of the reversed R trial relative to F (frames):", {c: v["lag"] for c, v in cells.items()})

# Pulse onset. The files do not give the delay of each frame. The onset is found on
# the forward trials: the first frame where the mean over unipolar cells of
# (upper - lower) / its baseline rises above the mean + 5 SD of frames 1-30.
uni = [c for c in cells if cells[c]["shape"] == "unipolar"]
bip = [c for c in cells if cells[c]["shape"] == "bipolar"]
d = np.nanmean([(cells[c]["uf"] - cells[c]["lf"]) for c in uni], axis=0)
base = d[1:31]
ONSET = int(np.nonzero(d > np.nanmean(base) + 5 * np.nanstd(base))[0][0])
print("pulse onset frame (forward trials):", ONSET, "=", ONSET * 25, "ns after the first frame")
# Methods 2.6: the mean of the 40 datapoints before the pulse is 100%
B0, B1 = ONSET - 40, ONSET

def pct(x):
    return (x / np.nanmean(x[B0:B1]) - 1) * 100

for c in cells:
    cells[c]["U"] = pct(cells[c]["u"])
    cells[c]["L"] = pct(cells[c]["l"])
    cells[c]["UF"], cells[c]["LF"] = pct(cells[c]["uf"]), pct(cells[c]["lf"])
    cells[c]["UR"], cells[c]["LR"] = pct(cells[c]["ur"]), pct(cells[c]["lr"])
print("baseline frames", B0, "to", B1 - 1)

In [ ]:
# Clock check: the response must start at the same frame in both trials of each
# cell (after alignment), and the pulse period must be 1.2 us = 48 frames. The
# autocorrelation of the mean unipolar upper-minus-lower trace, after the onset,
# gives the period.
t_ns = (np.arange(500) - ONSET) * 25.0
for c in cells:
    for tag in ["UF", "UR"]:
        x = cells[c][tag] - cells[c]["L" + tag[1]]
        b = x[B0:B1]
        on = np.nonzero(x[B0:] > np.nanmean(b) + 5 * np.nanstd(b))[0]
        cells[c]["on_" + tag] = int(on[0] + B0) if len(on) else None
print("onset frame per cell and trial:", {c: (cells[c]["on_UF"], cells[c]["on_UR"]) for c in cells})
x = np.nanmean([cells[c]["U"] - cells[c]["L"] for c in uni], axis=0)[ONSET:ONSET + 240]
x = x - x.mean()
ac = [np.sum(x[:-lag] * x[lag:]) for lag in range(20, 80)]
print("period of the unipolar response (frames):", 20 + int(np.argmax(ac)), "(expected 48 = 1200 ns / 25 ns)")
# which pole rises during the first pulse, per cell
for c in cells:
    w = slice(ONSET + 12, ONSET + 24)
    print(f"cell {c} ({cells[c]['shape']}): upper {np.nanmean(cells[c]['U'][w]):+.1f}%, lower {np.nanmean(cells[c]['L'][w]):+.1f}% (mean over 300-600 ns of pulse 1)")

Every cell's upper pole (smaller row index) rises and its lower pole falls during
the first pulse, so the field axis is the image's vertical axis in every file and
the orientation is the same in all cells, as expected for a fixed rig. FluoVolt
emission rises with depolarization (the dandiset's description gives its
sensitivity as dF/F per 100 mV, and the paper's Fig. 7 plots depolarization at the
cathode-facing pole as a positive change); the cathode-facing pole is therefore
the upper pole, and the anode-facing pole the lower one. The files do not say
where the electrodes were relative to the image, so this rests on the dye's sign.

In [ ]:
CATH, ANOD = "U", "L"
fig, axs = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
for ax, grp, name in [(axs[0], uni, "unipolar"), (axs[1], bip, "bipolar")]:
    for pole, col in [(CATH, "r"), (ANOD, "b")]:
        m = np.nanmean([cells[c][pole] for c in grp], axis=0)
        se = np.nanstd([cells[c][pole] for c in grp], axis=0, ddof=1) / np.sqrt(len(grp))
        ax.plot(t_ns, m, col, label={"U": "cathode-facing (upper)", "L": "anode-facing (lower)"}[pole])
        ax.fill_between(t_ns, m - se, m + se, color=col, alpha=0.25)
    for p in range(5):
        ax.axvspan(p * 1200, p * 1200 + 600, color="0.9", zorder=0)
    ax.set_title(f"{name}, {len(grp)} cells, mean +- s.e.m.")
    ax.set_xlabel("time from pulse onset (ns)")
axs[0].set_ylabel("% change of emission")
axs[0].legend()
allv = np.concatenate([np.nanmean([cells[c][p] for c in g], axis=0) for g in (uni, bip) for p in "UL"])
print("plotted range of the mean traces: %.1f to %.1f %%; x range %d to %d ns" % (np.nanmin(allv), np.nanmax(allv), t_ns[0], t_ns[-1]))
plt.show()

### fig7-unipolar-pole-polarization

> Unipolar 600 ns pulses depolarized the membrane quickly at the cathode-facing pole
> of the cell and hyperpolarized it in mirror image at the anode-facing pole.

Criterion: at the end of the first pulse (600 ns) the cathodic and anodic values
differ from zero in opposite directions (one-sample t-test across cells, p < 0.05),
and the anodic magnitude is within half to twice the cathodic. The value at 600 ns
is the frame 24 steps after the onset; the mean of frames 500-600 ns is a variant.

In [ ]:
def at(c, pole, t):
    return cells[c][pole][ONSET + int(round(t / 25))]

def win(c, pole, t0, t1):
    return np.nanmean(cells[c][pole][ONSET + int(t0 / 25):ONSET + int(t1 / 25) + 1])

def unipolar_polarization(key_cells):
    rows = []
    for label, f in [("value at 600 ns", lambda c, p: at(c, p, 600)), ("mean 500-600 ns", lambda c, p: win(c, p, 500, 600))]:
        ca = np.array([f(c, CATH) for c in key_cells])
        an = np.array([f(c, ANOD) for c in key_cells])
        tc, ta = stats.ttest_1samp(ca, 0), stats.ttest_1samp(an, 0)
        ratio = abs(an.mean()) / abs(ca.mean())
        ok = (tc.pvalue < 0.05) and (ta.pvalue < 0.05) and np.sign(ca.mean()) != np.sign(an.mean()) and 0.5 <= ratio <= 2
        rows.append({"reading": label, "cathodic mean %": ca.mean(), "cathodic p": tc.pvalue, "anodic mean %": an.mean(),
                     "anodic p": ta.pvalue, "|anodic|/|cathodic|": ratio, "n cells": len(key_cells), "criterion met": ok})
    return pd.DataFrame(rows)

r_pol = unipolar_polarization(uni)
print(r_pol.to_string())
print("per cell, value at 600 ns (cathodic, anodic):", {c: (round(at(c, CATH, 600), 1), round(at(c, ANOD, 600), 1)) for c in uni})
# variants of the pole regions (membrane threshold percentile, cone half-angle)
for key in ["traces_80_45", "traces_90_45", "traces_85_30", "traces_85_60"]:
    cv = build_cells(key)
    for c in cv:
        cv[c]["U"], cv[c]["L"] = pct(cv[c]["u"]), pct(cv[c]["l"])
    ca = np.array([cv[c]["U"][ONSET + 24] for c in uni]); an = np.array([cv[c]["L"][ONSET + 24] for c in uni])
    print(f"pole variant {key}: cathodic {ca.mean():+.1f}% (p={stats.ttest_1samp(ca,0).pvalue:.2g}), anodic {an.mean():+.1f}% (p={stats.ttest_1samp(an,0).pvalue:.2g}), ratio {abs(an.mean())/abs(ca.mean()):.2f}")

**Result: reproduced, resting on assumptions.** At the end of the first pulse the cathode-facing pole is +22.1% (p = 0.0012) and the anode-facing pole -20.7% (p = 0.0026) over 5 unipolar cells, opposite in sign, with a magnitude ratio of 0.94; the 500 to 600 ns mean and four variants of the pole regions give the same (ratios 0.90 to 1.05, all p <= 0.013). Every cell shows the same orientation. Which pole faces the cathode rests on FluoVolt brightening with depolarization, as the files do not record the electrode positions, and the pole regions are an assumption. The paper used 18 cells; 5 are in the dandiset.

### fig7-charging-time-constant

> Fitting the cathodic pole response to the first unipolar pulse gave a charging
> time constant of 200 ns and a maximum polarization of 31% emission change (425
> points, r = 0.93, p < 0.0001), and the discharge after it a time constant of 520
> ns from 31.5% (425 points, r = 0.85, p < 0.0001).

Charging, A (1 - exp(-t/tau)) over 0 to 600 ns of the first pulse; discharge,
A exp(-t/tau) over 600 to 1200 ns, t from the end of the pulse. Readings (a) the
points of every cell pooled, (b) the trace averaged over cells. The correlation
coefficient is that between the points and the fit.

In [ ]:
def fit_pulse(grp, pulse=0, pole=CATH, free=False):
    t0 = ONSET + pulse * 48
    tc = np.arange(0, 24) * 25.0          # frames 0..23 of the pulse: 0 to 575 ns
    td = np.arange(0, 24) * 25.0          # frames 24..47: 600 to 1175 ns, t from pulse end
    res = {}
    for reading in ["pooled", "mean"]:
        if reading == "pooled":
            vc = np.concatenate([cells[c][pole][t0:t0 + 24] for c in grp]); tcc = np.tile(tc, len(grp))
            vd = np.concatenate([cells[c][pole][t0 + 24:t0 + 48] for c in grp]); tdd = np.tile(td, len(grp))
        else:
            vc = np.nanmean([cells[c][pole][t0:t0 + 24] for c in grp], axis=0); tcc = tc
            vd = np.nanmean([cells[c][pole][t0 + 24:t0 + 48] for c in grp], axis=0); tdd = td
        okc, okd = np.isfinite(vc), np.isfinite(vd)
        if free:
            pc, fc = k.charge_fit_free(tcc[okc], vc[okc]); A, tau_c = pc[0], pc[2]
        else:
            pc, fc = k.charge_fit(tcc[okc], vc[okc]); A, tau_c = pc
        pd_, fd = k.discharge_fit(tdd[okd], vd[okd])
        rc = stats.pearsonr(vc[okc], fc); rd = stats.pearsonr(vd[okd], fd)
        res[reading] = {"tau_charge_ns": tau_c, "A_charge_%": A, "r_charge": rc[0], "p_charge": rc[1], "n_charge": int(okc.sum()),
                        "tau_discharge_ns": pd_[1], "A_discharge_%": pd_[0], "r_discharge": rd[0], "p_discharge": rd[1], "n_discharge": int(okd.sum())}
    return pd.DataFrame(res).T

r_fit1 = fit_pulse(uni, 0)
print(r_fit1.to_string())
exp_vals = {"tau_charge_ns": (200, 20), "A_charge_%": (31, 3.1), "tau_discharge_ns": (520, 52), "A_discharge_%": (31.5, 3.15)}
for reading in r_fit1.index:
    print(reading, {kk: bool(abs(r_fit1.loc[reading, kk] - v) <= tol) for kk, (v, tol) in exp_vals.items()})
# per cell
for c in uni:
    f1 = fit_pulse([c], 0).loc["mean"]
    print(f"cell {c}: tau charge {f1.tau_charge_ns:.0f} ns, A {f1['A_charge_%']:.1f}%, tau discharge {f1.tau_discharge_ns:.0f} ns, A {f1['A_discharge_%']:.1f}%")
# onset uncertainty: shifting the onset by one frame either way
for d_on in (-1, 1):
    ONSET += d_on
    f1 = fit_pulse(uni, 0).loc["pooled"]
    print(f"onset {d_on:+d} frame: tau charge {f1.tau_charge_ns:.0f} ns, tau discharge {f1.tau_discharge_ns:.0f} ns, A {f1['A_charge_%']:.1f} / {f1['A_discharge_%']:.1f}%")
    ONSET -= d_on

**Result: not reproduced.** Over the 5 unipolar cells the charging fit gives tau = 191 ns (within 180 to 220 ns) but a maximum of 23.2% (expected 31 +- 3.1%); the discharge fit gives tau = 613 ns (expected 520 +- 52 ns) from 22.7% (expected 31.5 +- 3.15%). The pooled points (120 per fit) and the mean trace give the same parameters; r = 0.84 and 0.64 for the pooled points (p < 1e-14), 0.998 and 0.997 for the mean trace. The discharge constant is 3.2 times the charging constant, against the paper's 2.5. Moving the onset by one frame gives charging tau 149 to 250 ns and discharge tau 615 to 627 ns. The dandiset holds 5 of the paper's 18 unipolar cells, and the per-cell amplitudes range from 14.5 to 31.3%, so the smaller amplitude may come from the subset or from the pole regions assumed here.

### fig7-later-pulses-same-tau-difference

> Fits to the later pulses of the unipolar train gave the same kind of difference
> between charging and discharging time constants as the first pulse.

The fits are repeated for pulses 2 to 5. Pulses 2 to 5 start from a charged
membrane, which A (1 - exp(-t/tau)) cannot fit; the charging fit for them lets the
starting value free, V0 + (A - V0)(1 - exp(-t/tau)), and the plain form is shown
too. The discharge after pulse 5 is fitted over 600 ns like the others.

In [ ]:
ratio1 = {r: r_fit1.loc[r, "tau_discharge_ns"] / r_fit1.loc[r, "tau_charge_ns"] for r in r_fit1.index}
later = []
for p in range(1, 5):
    for free in (True, False):
        f = fit_pulse(uni, p, free=free)
        for r in f.index:
            rr = f.loc[r, "tau_discharge_ns"] / f.loc[r, "tau_charge_ns"]
            later.append({"pulse": p + 1, "charging form": "free start" if free else "A(1-exp)", "reading": r,
                          "tau charge": f.loc[r, "tau_charge_ns"], "tau discharge": f.loc[r, "tau_discharge_ns"], "ratio": rr,
                          "discharge > charge": rr > 1, "ratio in 2.25-2.75": 2.25 <= rr <= 2.75,
                          "ratio within 10% of pulse 1": abs(rr - ratio1[r]) <= 0.1 * ratio1[r]})
later = pd.DataFrame(later)
print("pulse 1 ratio (discharge / charge):", {r: round(v, 2) for r, v in ratio1.items()})
print(later.round(3).to_string())

**Result: partially reproduced.** With the free-start charging fit, the discharge constant exceeds the charging constant for all four later pulses (ratios 2.97, 3.72, 2.91, 2.89). None is within 2.25 to 2.75; within 10% of the first pulse's ratio (3.21, so 2.89 to 3.53) are pulses 2 and 4, and pulse 5 falls just below (2.887). With the plain A(1 - exp(-t/tau)) form the ratios are 8 to 11, as that form cannot fit a pulse that starts from a charged membrane.

### fig7-bipolar-overshoot

> Bipolar pulses caused the same initial polarization as unipolar ones during the
> first phase, followed by a faster discharge and an overshoot into the opposite
> polarity.

In [ ]:
cu = np.array([at(c, CATH, 600) for c in uni]); cb = np.array([at(c, CATH, 600) for c in bip])
t_i = stats.ttest_ind(cb, cu)
gap = abs(cb.mean() - cu.mean()) > (stats.sem(cb) + stats.sem(cu))
print(f"(i) cathodic value at 600 ns: bipolar {cb.mean():.1f}% (n={len(cb)}), unipolar {cu.mean():.1f}% (n={len(cu)}); "
      f"difference {cb.mean()-cu.mean():+.1f}%, unpaired t-test p = {t_i.pvalue:.3g}; error-bar gap rule says different: {gap}")
mu = np.nanmean([cells[c][CATH] for c in uni], axis=0); mb_ = np.nanmean([cells[c][CATH] for c in bip], axis=0)
def first_return(m, frac):
    pk = m[ONSET + 24]
    after = np.nonzero(m[ONSET + 24:ONSET + 48] <= frac * pk)[0]
    return after[0] * 25 if len(after) else None
for frac in (0.5, 0.1, 0.0):
    print(f"(ii) time after the end of the first phase for the mean trace to fall to {frac:.0%} of its 600 ns value: bipolar {first_return(mb_, frac)} ns, unipolar {first_return(mu, frac)} ns (None: not within 600 ns)")
seg = mb_[ONSET + 24:ONSET + 48]
imin = int(np.nanargmin(seg)); vals = np.array([cells[c][CATH][ONSET + 24 + imin] for c in bip])
t_iii = stats.ttest_1samp(vals, 0)
print(f"(iii) bipolar mean minimum in 600-1200 ns: {seg[imin]:.1f}% at {600 + imin*25} ns; cell values {np.round(vals,1).tolist()}, one-sample t p = {t_iii.pvalue:.3g}")
print("per bipolar cell, minimum in 600-1200 ns:", {c: round(float(np.nanmin(cells[c][CATH][ONSET + 24:ONSET + 48])), 1) for c in bip})

**Result: reproduced.** (i) At 600 ns the cathodic pole is 21.5% after bipolar pulses (4 cells) and 22.1% after unipolar pulses (5 cells), p = 0.855, and the error-bar gap rule finds no difference. (ii) After the first phase the bipolar mean falls to half its value in 150 ns against 475 ns for unipolar pulses, and to zero in 325 ns, which the unipolar trace does not reach before the next pulse. (iii) The bipolar mean reaches -8.1% at 1175 ns; the four cell values (-6.7 to -9.9%) differ from zero, p = 0.0012. Four of the paper's 17 bipolar cells are in the dandiset.

### fig7-bipolar-alternation

> In 833 kHz trains, bipolar pulses caused the membrane potential to alternate
> repeatedly between positive and negative values.

In [ ]:
alt = []
for p in range(5):
    t0 = ONSET + p * 48
    end1 = np.array([cells[c][CATH][t0 + 24] for c in bip])
    mseg = np.nanmean([cells[c][CATH][t0 + 24:t0 + 48] for c in bip], axis=0)
    im = int(np.nanargmin(mseg))
    mins = np.array([cells[c][CATH][t0 + 24 + im] for c in bip])
    a, b = stats.ttest_1samp(end1, 0), stats.ttest_1samp(mins, 0)
    alt.append({"pulse": p + 1, "end of phase 1 %": end1.mean(), "p": a.pvalue, "phase 2 minimum %": mins.mean(), "at ns": 600 + im * 25,
                "p ": b.pvalue, "both significant, right signs": end1.mean() > 0 and mins.mean() < 0 and a.pvalue < 0.05 and b.pvalue < 0.05})
alt = pd.DataFrame(alt)
print(alt.round(4).to_string())
print("pulses meeting the criterion:", int(alt.iloc[:, -1].sum()), "of 5; n =", len(bip), "cells")

**Result: reproduced.** For each of the five pulses, the cathodic pole is positive at the end of the first phase (19.3 to 21.5%, p <= 0.0001) and negative at its second phase minimum (-8.1 to -9.1%, p <= 0.0013), over 4 bipolar cells.

### fig7-summation-without-buildup

> With 600 ns unipolar pulses at 833 kHz, each pulse charged the membrane to the
> same peak, with no stepwise build-up; the pulses merged into a continuous
> sawtooth polarization.

In [ ]:
peaks = np.array([[cells[c][CATH][ONSET + p * 48 + 24] for p in range(5)] for c in uni])
pre = np.array([[cells[c][CATH][ONSET + p * 48 - 1] for p in range(1, 5)] for c in uni])
rows = []
for p in range(1, 5):
    tu = stats.ttest_ind(peaks[:, p], peaks[:, 0]); tp = stats.ttest_rel(peaks[:, p], peaks[:, 0])
    tpre = stats.ttest_1samp(pre[:, p - 1], 0)
    rows.append({"pulse": p + 1, "peak %": peaks[:, p].mean(), "peak - first": peaks[:, p].mean() - peaks[:, 0].mean(),
                 "unpaired p": tu.pvalue, "paired p (variant)": tp.pvalue, "value before pulse %": pre[:, p - 1].mean(), "p vs 0": tpre.pvalue})
summ = pd.DataFrame(rows)
print("first peak: %.1f%%" % peaks[:, 0].mean())
print(summ.round(4).to_string())
# peak taken as the maximum within each pulse (variant)
pk2 = np.array([[np.nanmax(cells[c][CATH][ONSET + p * 48:ONSET + p * 48 + 25]) for p in range(5)] for c in uni])
print("variant, maximum within each pulse:", pk2.mean(0).round(1), "unpaired p vs first:", [round(stats.ttest_ind(pk2[:, p], pk2[:, 0]).pvalue, 3) for p in range(1, 5)])

**Result: reproduced.** The peaks of pulses 2 to 5 (23.4 to 24.3%) do not differ from the first (22.1%) by the unpaired test (p = 0.61 to 0.75), and the trace before each of pulses 2 to 5 stays above rest (8.8 to 9.9%, p = 0.009 to 0.017), over 5 unipolar cells. The paired variant finds the later peaks 1.4 to 2.2 points higher, with p < 0.05 for pulses 2 to 4 (p = 0.014 to 0.040); the claim's absence test is the unpaired one.

## Data check: dandiset 000489 (dish images, Fig. 2)

Each file holds one stitched image of a whole dish, 22323 x 22323 pixels with two
channels. The images are streamed in bands and reduced to 7 x 7 pixel block means
as they load (4 dishes at a time); everything below works on the block means.

In [ ]:
dish_assets = k.list_assets("000489")
from concurrent.futures import ProcessPoolExecutor
import time
t0 = time.time()
with ProcessPoolExecutor(4) as ex:
    dishes = list(ex.map(k.process_dish, dish_assets))
print(f"loaded {len(dishes)} images in {time.time() - t0:.0f} s")
dm = pd.DataFrame([{kk: d[kk] for kk in ["path", "subject_id", "session_id", "session_start_time", "shape"]} for d in dishes])
print(dm.to_string())
print("series description (first file):", dishes[0]["description"])

In [ ]:
import re
def amp_of(d):
    m = re.match(r"Fig2_(\d+)_", d["subject_id"] + "_")
    return int(m.group(1)) if m else None
for d in dishes:
    d["amp"] = amp_of(d)
fig2 = [d for d in dishes if d["amp"] is not None]
print("Fig. 2 dishes per second phase amplitude (%):", pd.Series([d["amp"] for d in fig2]).value_counts().sort_index().to_dict())
print("other files:", [d["subject_id"] for d in dishes if d["amp"] is None])
for d in dishes:
    fp = d.get("footprints")
    line = f"{d['subject_id']:15s} ch0 median {d['channel_median'][0]:6.0f} max {d['channel_max'][0]:7.0f} | ch1 median {d['channel_median'][1]:6.0f} max {d['channel_max'][1]:7.0f} | digests {d['digest']}"
    if fp is not None:
        c = fp["centres"]
        sep = np.linalg.norm(c[1] - c[0]) * k.UM_PER_BLOCK / 1000
        line += f" | footprints {sep:.2f} mm apart, radii {np.round(fp['radius_px'] * k.UM_PER_BLOCK / 1000, 2).tolist()} mm | centre/far ch0 {d['centre_over_far'][0]:.2f} ch1 {d['centre_over_far'][1]:.2f}"
    print(line)
seps = [np.linalg.norm(d["footprints"]["centres"][1] - d["footprints"]["centres"][0]) * k.UM_PER_BLOCK / 1000 for d in fig2]
print(f"electrode centre spacing at 0.65 um/pixel: {np.min(seps):.2f} to {np.max(seps):.2f} mm (median {np.median(seps):.2f}); Methods 2.2 gives 3 mm side to side for 1.5 mm electrodes, 4.5 mm centre to centre")
# images deposited twice
from collections import defaultdict
for ch in range(2):
    groups = defaultdict(list)
    for d in dishes:
        groups[d["digest"][ch]].append(d["subject_id"])
    print(f"channel {ch}: images shared by more than one file:", [v for v in groups.values() if len(v) > 1])

Channel index 1 is three to six times brighter between the electrodes than 3 mm
away along bb', and index 0 is flat, so index 1 is YO-PRO-1 and index 0 MemBrite. The
series description names "Channel1" for both filter sets and for MemBrite, which
does not say which index holds which dye. The electrode spacing measured at
0.65 um per pixel agrees with the 4.5 mm of Methods 2.2, which supports the pixel
size (the files give none).

The YP channel of sub-Fig2-0-03 (unipolar) and sub-Fig2-25-04 (25% bipolar) is the
same image, block for block, while their MemBrite channels differ. One recording
was deposited under two dishes, and nothing says which dish it belongs to. The
main analysis counts it once by leaving out both dishes (unipolar and 25% then have
2 dishes each); the variants assign the shared image to the unipolar group or to
the 25% group.

In [ ]:
fig, axs = plt.subplots(3, 6, figsize=(18, 9))
for ax, d in zip(axs.ravel(), dishes):
    im = d["thumb_yp"]
    ax.imshow(im, vmin=np.percentile(im, 2), vmax=np.percentile(im, 99.5), cmap="gray")
    if d.get("footprints") is not None:
        for c in d["footprints"]["centres"]:
            ax.add_patch(plt.Circle((c[1] / 8, c[0] / 8), 750 / k.UM_PER_BLOCK / 8, fill=False, color="r"))
        mid, ua, ub = k.dish_frame(d["footprints"]["centres"])
        p = np.array([mid + ub * o * 1000 / k.UM_PER_BLOCK for o in (-3, 3)]) / 8
        ax.plot(p[:, 1], p[:, 0], "y-", lw=1)
    ax.set_title(d["subject_id"], fontsize=8)
    ax.axis("off")
plt.suptitle("YP channel; red circles: footprints found (1.5 mm), yellow: bb' line, +-3 mm")
plt.tight_layout()
plt.show()

### Field model and ROI

The field of each ROI comes from the two-cylinder solution in the helper (the
geometry of Methods 2.3, 5.75 kV). It gives 15.38 kV/cm at the centre, against the
15.1 kV/cm the paper's finite element model gives there (Results 3.1 and Fig. 2C
legend): 1.8% higher. The named fields 15.1, 8.7 and 6.7 kV/cm are taken at the ROI
whose model field is closest, scaled by 15.1/15.38 (the paper's value at the
centre); this scaling does not change which ROI is closest.

In [ ]:
offs = np.round(np.arange(0, 3.01, 0.2), 1)
fields = np.array([k.roi_mean_field(0, o) for o in offs])
scaled = fields * 15.1 / fields[0]
print(pd.DataFrame({"offset along bb' (mm)": offs, "model field (kV/cm)": fields.round(2), "scaled to 15.1 at centre": scaled.round(2)}).to_string(index=False))
def nearest(kv):
    return float(offs[np.argmin(abs(scaled - kv))])
print("ROI closest to 15.1, 8.7, 6.7 kV/cm (offset, mm):", nearest(15.1), nearest(8.7), nearest(6.7))

In [ ]:
def long_table(key="rois", assign=None):
    """One row per dish and ROI. assign: None (leave out the duplicated pair),
    0 or 25 (use the shared YP image once, under that amplitude)."""
    rows = []
    for d in fig2:
        if d["subject_id"] in ("Fig2_0_03", "Fig2_25_04"):
            if assign is None:
                continue
            if (assign == 0) != (d["subject_id"] == "Fig2_0_03"):
                continue
        for r in d[key]:
            rows.append({"dish": d["subject_id"], "amp": d["amp"], **r})
    t = pd.DataFrame(rows)
    t["field"] = t.field_kv_cm * 15.1 / fields[0]
    t["abs_off"] = t.offset_mm.abs().round(1)
    return t

def per_dish(t, reading="a"):
    """Dish value per distance from the centre: (a) mean of the two ROI at +-offset,
    (b+) the positive side only, (b-) the negative side only."""
    if reading == "a":
        s = t
    elif reading == "b+":
        s = t[t.offset_mm >= 0]
    else:
        s = t[t.offset_mm <= 0]
    g = s.groupby(["amp", "dish", "abs_off"]).agg(YP=("YP", "mean"), field=("field", "mean")).reset_index()
    return g

T = {None: long_table(), 0: long_table(assign=0), 25: long_table(assign=25)}
D = {(a, r): per_dish(T[a], r) for a in T for r in ["a", "b+", "b-"]}
main = D[(None, "a")]
summary = main.groupby(["amp", "abs_off"]).YP.agg(["mean", "sem", "count"]).unstack(0)
print("Mean YP (a.u.) per ROI distance and amplitude, reading (a), duplicated pair left out:")
print(summary["mean"].round(0).assign(field=np.round(scaled, 2)).to_string())
print("dishes per amplitude:", main.groupby("amp").dish.nunique().to_dict())
print("YP range over dish values: %.0f to %.0f a.u." % (main.YP.min(), main.YP.max()))

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(20, 3.8))
for ax, a in zip(axs, [0, 25, 50, 75, 100]):
    s = main[main.amp == a].groupby("abs_off").agg(m=("YP", "mean"), e=("YP", "sem"), f=("field", "mean"))
    ax.errorbar(s.f, s.m, s.e, fmt="o-")
    ax.set_title(f"{a}% second phase (n={main[main.amp == a].dish.nunique()})")
    ax.set_xlabel("field (kV/cm)")
axs[0].set_ylabel("YP (a.u.)")
plt.tight_layout()
plt.show()

### fig2a-uptake-rises-with-field

> For unipolar 600 ns pulses and for bipolar 600+600 ns pulses with every second
> phase amplitude, a train of 5 pulses at 1 Hz evoked YP uptake that increased with
> the local electric field strength.

Strongest-field ROI (centre) against weakest-field ROI (3.0 mm, 5.0 kV/cm, the
outermost ROI of the set used here), dish values, two-sided t-test unpaired and
paired. The ROI set's outer limit is not given by the paper (its figure range is in
the PDF only); the ROI at 2.4 mm (6.6 kV/cm) is shown as a variant weakest ROI.

In [ ]:
def rises(g, weak=3.0):
    out = []
    for a in [0, 25, 50, 75, 100]:
        s = g[g.amp == a].pivot(index="dish", columns="abs_off", values="YP")
        hi, lo = s[0.0], s[weak]
        tu = stats.ttest_ind(hi, lo); tp = stats.ttest_rel(hi, lo)
        m = s.mean()
        rho = stats.spearmanr(g[g.amp == a].groupby("abs_off").field.mean(), m)
        out.append({"amp %": a, "n": len(s), "YP strongest": hi.mean(), "YP weakest": lo.mean(), "unpaired p": tu.pvalue,
                    "paired p": tp.pvalue, "higher & p<0.05 (unpaired)": hi.mean() > lo.mean() and tu.pvalue < 0.05,
                    "higher & p<0.05 (paired)": hi.mean() > lo.mean() and tp.pvalue < 0.05, "Spearman rho (means vs field)": rho[0], "rho p": rho[1]})
    return pd.DataFrame(out)

for (a, r), g in D.items():
    for weak in (3.0, 2.4):
        res = rises(g, weak)
        tag = f"duplicate {'left out' if a is None else 'as ' + str(a) + '%'}, reading ({r}), weakest ROI at {weak} mm"
        print(tag, "| amplitudes passing, unpaired:", int(res.iloc[:, 6].sum()), "paired:", int(res.iloc[:, 7].sum()))
        if (a, r, weak) == (None, "a", 3.0):
            main_rises = res
print()
print(main_rises.round(4).to_string())

**Result: reproduced, resting on an assumption.** With the two dishes that share one YP image left out, YP is higher at the strongest field than at the weakest ROI for all five amplitudes with p < 0.05 by the unpaired test (p = 0.0001 to 0.044), and the Spearman correlation of mean YP with field is 0.96 to 0.99. By the paired test 3 of 5 pass (unipolar and 25%, with 2 dishes each, give p = 0.12 and 0.13). Every ROI reading and the weaker ROI at 2.4 mm give the same counts. Assigning the shared image to the unipolar or the 25% group gives 4 of 5 (unpaired). The outcome takes the unpaired test, the two-sample test that the Methods' name denotes, as an assumption.

### fig2a-vertical-scales

> The vertical scale of the Fig. 2A panels goes from 1200 a.u. for unipolar pulses
> down to only 250 a.u. for bipolar pulses with a 75% second phase.

The image values are 16-bit counts with no conversion (conversion 1.0, offset 0 in
the files); nothing shows that they were rescaled on export, and the footprint
levels (about 180 to 300 counts) are plausible camera counts. They are taken as
the units the paper measured, which the data are consistent with but cannot prove.

In [ ]:
for (a, r) in [(None, "a"), (None, "b+"), (None, "b-"), (0, "a"), (25, "a")]:
    g = D[(a, r)]
    s = g.groupby(["amp", "abs_off"]).YP.agg(["mean", "sem"])
    top = (s["mean"] + s["sem"]).groupby(level=0).max()
    mx = s["mean"].groupby(level=0).max()
    print(f"duplicate {'left out' if a is None else 'as ' + str(a) + '%'}, reading ({r}): largest mean+sem per amplitude {top.round(0).to_dict()};"
          f" largest mean {mx.round(0).to_dict()}; unipolar <= 1200: {top[0] <= 1200}; 75% <= 250: {top[75] <= 250}; unipolar mean > 250: {mx[0] > 250};"
          f" ratio of largest means 0%/75% {mx[0] / mx[75]:.2f} (paper's axes 4.8)")

**Result: not reproduced.** The largest unipolar mean is 651 a.u. (mean + s.e.m. 754, below 1200, and above 250), but for 75% bipolar pulses the largest mean + s.e.m. is 339 a.u., above 250, under every reading (311 to 372). The ratio of the largest unipolar and 75% means is 2.3 to 2.5, against the 4.8 of the paper's axes. This takes the image counts as the paper's units, which the files are consistent with but do not state.

### fig2b-exponential-same-slope

> YP uptake increased exponentially with the field strength, and the second phase
> did not change the slope of the exponential fits on a semi-log scale.

Fits to the mean values over ROI at or above the cutoff (8 kV/cm; 6 and 7 as
variants): YP = A exp(kE) by nonlinear least squares, against a straight line.
Per-dish fits give a slope k per dish; each bipolar amplitude is compared with the
unipolar slopes by an unpaired two-sided t-test. Variant: linear regression of log
YP on field (ROI with YP <= 0 dropped).

In [ ]:
from scipy.optimize import curve_fit
expf = lambda E, A, kk: A * np.exp(kk * E)

def fit_exp(E, Y):
    try:
        p, cov = curve_fit(expf, E, Y, p0=[max(np.mean(Y), 1) * np.exp(-0.2 * np.mean(E)), 0.2], maxfev=20000)
        return p, np.sqrt(np.diag(cov)), np.sum((Y - expf(E, *p)) ** 2)
    except Exception as e:
        return None, None, np.nan

def fig2b(g, cut, method="nls"):
    rows, slopes = [], {}
    for a in [0, 25, 50, 75, 100]:
        s = g[(g.amp == a) & (g.field >= cut)]
        m = s.groupby("abs_off").agg(E=("field", "mean"), Y=("YP", "mean"))
        lin = np.polyfit(m.E, m.Y, 1); rss_lin = np.sum((m.Y - np.polyval(lin, m.E)) ** 2)
        p, se, rss_exp = fit_exp(m.E.values, m.Y.values)
        ks = []
        for dish, sd in s.groupby("dish"):
            if method == "nls":
                pp, _, _ = fit_exp(sd.field.values, sd.YP.values)
                ks.append(pp[1] if pp is not None else np.nan)
            else:
                ok = sd.YP > 0
                ks.append(np.polyfit(sd.field[ok], np.log(sd.YP[ok]), 1)[0] if ok.sum() >= 3 else np.nan)
        slopes[a] = np.array(ks)
        rows.append({"amp %": a, "ROI": len(m), "k of fit to means (1/(kV/cm))": p[1] if p is not None else np.nan,
                     "k se": se[1] if se is not None else np.nan, "RSS exp": rss_exp, "RSS line": rss_lin, "exp better": rss_exp < rss_lin,
                     "per-dish k": np.round(slopes[a], 3).tolist()})
    out = pd.DataFrame(rows)
    out["k ratio to unipolar"] = out["k of fit to means (1/(kV/cm))"] / out.loc[0, "k of fit to means (1/(kV/cm))"]
    ps = {}
    for a in [25, 50, 75, 100]:
        x, y = slopes[a][np.isfinite(slopes[a])], slopes[0][np.isfinite(slopes[0])]
        ps[a] = stats.ttest_ind(x, y).pvalue if len(x) > 1 and len(y) > 1 else np.nan
    out["p slope vs unipolar"] = out["amp %"].map(ps)
    return out

for cut in (8, 7, 6):
    for method in ("nls", "log-linear"):
        for key in [(None, "a"), (0, "a"), (25, "a")]:
            r = fig2b(D[key], cut, method)
            nexp = int(r["exp better"].sum()); ndiff = int((r["p slope vs unipolar"] < 0.05).sum())
            print(f"cutoff {cut} kV/cm, per-dish {method}, duplicate {'left out' if key[0] is None else 'as ' + str(key[0]) + '%'}: exp better for {nexp}/5;"
                  f" bipolar slopes differing from unipolar (p<0.05): {ndiff}/4; p = {r['p slope vs unipolar'].dropna().round(3).tolist()}")
            if (cut, method, key) == (8, "nls", (None, "a")):
                main_2b = r
print()
print(main_2b.round(4).to_string())

**Result: partially reproduced.** At the 8 kV/cm cutoff the exponential fits the means better than a line for 3 of 5 amplitudes, short of the four the criterion needs; at 7 kV/cm for 4 of 5 and at 6 kV/cm for 5 of 5. No bipolar slope differs from the unipolar slope by the per-dish nonlinear fits at any cutoff (p = 0.08 to 1.0 over every cutoff and assignment of the shared image; 0.12 to 0.80 at 8 kV/cm). The slopes of the fits to the means are 0.19 to 0.28 per kV/cm, 0.72 to 1.07 times the unipolar 0.26. With the log-linear variant the bipolar slopes differ at 6 and 7 kV/cm (4 of 4, p = 0.004 to 0.042) when the shared image is left out, and not when it is assigned to the unipolar group.

### fig2c-max-inhibition-50-75

> The strongest inhibition of YP uptake, to about 20% of the uptake after unipolar
> pulses, was reached with a second phase of 50 to 75% of the first (p < 0.01).

Percent of unipolar YP at each ROI with field 8.7 to 15.1 kV/cm (scaled fields;
the ROI at 0 to 1.8 mm). Ordering: the lower of 50% and 75% below 25% and 100% at
every such field. "~20%": (a) the mean over those fields for the better of 50 and
75%, (b) the lowest percent at any of them. Tests, unpaired: (i) 50% and 75% dish
YP against unipolar, (ii) 50% and 75% against 25% and 100%, at each field.

In [ ]:
rng = sorted(main.abs_off[(main.field >= 8.7 - 1e-9) & (main.field <= 15.1 + 1e-9)].unique())
print("ROI in 8.7 to 15.1 kV/cm (offsets, mm):", rng, "| scaled fields:", [round(float(scaled[list(offs).index(o)]), 2) for o in rng])

def fig2c(g):
    m = g.groupby(["amp", "abs_off"]).YP.mean().unstack(0)
    pc = (m.div(m[0], axis=0) * 100).loc[rng]
    order = ((pc[[50, 75]].min(axis=1) < pc[25]) & (pc[[50, 75]].min(axis=1) < pc[100]))
    best = 50 if pc[50].mean() <= pc[75].mean() else 75
    ra, rb = pc[best].mean(), pc[[50, 75]].min(axis=1).min()
    ti, tii = [], []
    for o in rng:
        v = {a: g[(g.amp == a) & (g.abs_off == o)].YP.values for a in [0, 25, 50, 75, 100]}
        ti.append({a: stats.ttest_ind(v[a], v[0]).pvalue for a in (50, 75)})
        tii.append({(a, b): stats.ttest_ind(v[a], v[b]).pvalue for a in (50, 75) for b in (25, 100)})
    ti, tii = pd.DataFrame(ti, index=rng), pd.DataFrame(tii, index=rng)
    return pc, order, best, ra, rb, ti, tii

for key in [(None, "a"), (0, "a"), (25, "a"), (None, "b+"), (None, "b-")]:
    pc, order, best, ra, rb, ti, tii = fig2c(D[key])
    i_all = bool((ti.min(axis=1) < 0.01).all()); ii_all = bool(((tii[[(50, 25), (50, 100)]].max(axis=1) < 0.01) | (tii[[(75, 25), (75, 100)]].max(axis=1) < 0.01)).all())
    print(f"duplicate {'left out' if key[0] is None else 'as ' + str(key[0]) + '%'}, reading ({key[1]}): ordering holds at {int(order.sum())}/{len(order)} fields;"
          f" best {best}%: (a) mean {ra:.1f}%, (b) lowest {rb:.1f}%; (i) p<0.01 for 50 or 75% vs unipolar at {int((ti.min(axis=1) < 0.01).sum())}/{len(rng)} fields;"
          f" (ii) p<0.01 against both 25 and 100% at {int(((tii[[(50, 25), (50, 100)]].max(axis=1) < 0.01) | (tii[[(75, 25), (75, 100)]].max(axis=1) < 0.01)).sum())}/{len(rng)} fields")
    if key == (None, "a"):
        main_2c = (pc, order, best, ra, rb, ti, tii)
pc, order, best, ra, rb, ti, tii = main_2c
print()
print("percent of unipolar, main reading:")
print(pc.round(1).assign(ordering=order).to_string())
print("(i) p, 50% and 75% against unipolar:")
print(ti.round(4).to_string())
print("(ii) p:")
print(tii.round(4).to_string())

**Result: partially reproduced.** At every ROI from 8.78 to 15.1 kV/cm the lower of the 50% and 75% values is below the 25% and 100% values (10 of 10). The best amplitude, 50%, gives 49.3% of unipolar on average (reading a) and 35.1% at its lowest (reading b), both outside 20 +- 2%. By test (ii) the better of 50 or 75% differs from both 25% and 100% with p < 0.01 at 7 of 10 fields; by test (i), against unipolar, at 3 of 10. The test is counted as holding under (ii) at most fields. The 25% and 100% pulses give 64 to 168% of the unipolar value, so in these dishes only 50 and 75% inhibit.

### fig2c-inhibition-flat-above-8-7

> Between 8.7 and 15.1 kV/cm the degree of inhibition by the second phase did not
> depend on the field strength.

Each bipolar dish value divided by the unipolar mean at the same ROI; 8.7 kV/cm
(ROI at 1.8 mm) against 15.1 kV/cm (centre), unpaired two-sided t-test, p < 0.05.

In [ ]:
def ratio_table(g):
    um = g[g.amp == 0].groupby("abs_off").YP.mean()
    r = g[g.amp > 0].copy()
    r["pct"] = r.YP / r.abs_off.map(um) * 100
    return r

o87, o151, o67 = nearest(8.7), nearest(15.1), nearest(6.7)
for key in [(None, "a"), (0, "a"), (25, "a"), (None, "b+"), (None, "b-")]:
    r = ratio_table(D[key])
    rows = []
    for a in [25, 50, 75, 100]:
        x = r[(r.amp == a) & (r.abs_off == o87)].set_index("dish").pct
        y = r[(r.amp == a) & (r.abs_off == o151)].set_index("dish").pct
        rows.append({"amp %": a, "% at 8.7": x.mean(), "% at 15.1": y.mean(), "difference": x.mean() - y.mean(),
                     "unpaired p": stats.ttest_ind(x, y).pvalue, "paired p (variant)": stats.ttest_rel(x, y.loc[x.index]).pvalue})
    rows = pd.DataFrame(rows)
    # every pair of ROI in the range (variant)
    npairs = nsig = 0
    for a in [25, 50, 75, 100]:
        for i, o1 in enumerate(rng):
            for o2 in rng[i + 1:]:
                npairs += 1
                nsig += stats.ttest_ind(r[(r.amp == a) & (r.abs_off == o1)].pct, r[(r.amp == a) & (r.abs_off == o2)].pct).pvalue < 0.05
    print(f"duplicate {'left out' if key[0] is None else 'as ' + str(key[0]) + '%'}, reading ({key[1]}): amplitudes with p<0.05 (unpaired): {int((rows['unpaired p'] < 0.05).sum())}/4;"
          f" every-pair variant: {nsig}/{npairs} pairs differ")
    if key == (None, "a"):
        print(rows.round(4).to_string())

**Result: reproduced.** No bipolar amplitude differs between 8.78 and 15.1 kV/cm by the unpaired test (p = 0.07 to 0.83). The differences are large for 50% (67.8% against 35.1%, p = 0.072) and 75% (91.5% against 40.0%, p = 0.15), with 3 to 5 dishes. Comparing every pair of ROI in the range, 19 of 180 pairs differ at p < 0.05; reading (b) finds one amplitude differing on either side.

### fig2c-weaker-at-6-7

> Inhibition was weaker at 6.7 kV/cm than at the higher field strengths, with
> p < 0.01 for the 50% bipolar pulses and p < 0.05 for the 100% bipolar pulses.

Percent of the unipolar mean at the ROI closest to 6.7 kV/cm (2.4 mm) against (i)
each ROI from 8.7 to 15.1 kV/cm and (ii) those ROI pooled; unpaired and paired.

In [ ]:
for key in [(None, "a"), (0, "a"), (25, "a"), (None, "b+"), (None, "b-")]:
    r = ratio_table(D[key])
    lines = []
    verdict = {}
    for a, thr in [(50, 0.01), (100, 0.05)]:
        x = r[(r.amp == a) & (r.abs_off == o67)].set_index("dish").pct
        higher = [r[(r.amp == a) & (r.abs_off == o)].set_index("dish").pct for o in rng]
        each_u = [stats.ttest_ind(x, h).pvalue for h in higher]
        each_p = [stats.ttest_rel(x, h.loc[x.index]).pvalue for h in higher]
        pooled = pd.concat(higher)
        pool_u = stats.ttest_ind(x, pooled).pvalue
        pool_p = stats.ttest_rel(x, pd.concat(higher, axis=1).mean(axis=1).loc[x.index]).pvalue
        above_all = bool(all(x.mean() > h.mean() for h in higher))
        sig = (max(each_u) < thr) or (pool_u < thr) or (max(each_p) < thr) or (pool_p < thr)
        verdict[a] = (above_all, sig)
        lines.append(f"{a}%: at 6.7 kV/cm {x.mean():.1f}% vs higher fields {min(h.mean() for h in higher):.1f} to {max(h.mean() for h in higher):.1f}% (pooled {pooled.mean():.1f}%);"
                     f" higher than every higher field: {above_all}; (i) unpaired p max {max(each_u):.3g}, paired p max {max(each_p):.3g}; (ii) pooled unpaired p {pool_u:.3g}, paired {pool_p:.3g}; p<{thr} under some reading: {sig}")
    print(f"duplicate {'left out' if key[0] is None else 'as ' + str(key[0]) + '%'}, reading ({key[1]}):")
    for l in lines:
        print("   ", l)

**Result: reproduced, resting on an assumption.** At 6.63 kV/cm the 50% pulses give 198% of the unipolar mean against 35 to 68% at the higher fields, and the 100% pulses 283% against 93 to 139%. With the unpaired test, 50% differs from the pooled higher fields with p = 1.5e-09 (reading ii; against each field, p up to 0.082), and 100% differs from each higher field with p <= 0.022 and from the pooled ones with p = 2.6e-09. With the paired test, 100% passes (p <= 0.035 against each) and 50% does not (p = 0.07 pooled). The outcome takes the unpaired test, as for fig2a-uptake-rises-with-field. The unipolar mean at that ROI is low (19 a.u.), so the percentages there are large and unstable.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Dataset | Outcome |
|---|---|---|
| fig2a-uptake-rises-with-field | 000489 | reproduced (assumed: unpaired test) |
| fig2a-vertical-scales | 000489 | not reproduced |
| fig2b-exponential-same-slope | 000489 | partially reproduced |
| fig2c-max-inhibition-50-75 | 000489 | partially reproduced |
| fig2c-inhibition-flat-above-8-7 | 000489 | reproduced |
| fig2c-weaker-at-6-7 | 000489 | reproduced (assumed: unpaired test) |
| fig7-unipolar-pole-polarization | 000448 | reproduced (assumed: pole regions, cathode side) |
| fig7-charging-time-constant | 000448 | not reproduced |
| fig7-later-pulses-same-tau-difference | 000448 | partially reproduced |
| fig7-bipolar-overshoot | 000448 | reproduced |
| fig7-bipolar-alternation | 000448 | reproduced |
| fig7-summation-without-buildup | 000448 | reproduced |
| Figs. 3 to 6 (12 claims) | 000489 | blocked: dataset missing (likely in 000548 to 000550) |
| fig1e-heating | 000489 | blocked: data missing |

**How the analysis differs from the paper's.** The field of each ROI comes from an exact two
dimensional solution for the electrode geometry of Methods 2.3, not the paper's finite element
model; it gives 15.38 kV/cm at the centre against the paper's 15.1, and the fields are scaled to
15.1 there. ROI are contiguous 200 um squares along bb' from the centre to 3 mm, averaged over both
sides (one side as variants); the paper does not give their number or spacing. YP0 and MB0 are
taken in a 0.2 mm disc at the centre of each footprint. The two dishes that share one YP image are
left out, so the unipolar and 25% groups have 2 dishes each. The Fig. 7 analysis uses 5 of the
paper's 18 unipolar cells and 4 of its 17 bipolar cells; the pole regions are drawn here, the pulse
onset is found from the data, and the cathode-facing pole is taken to be the one that brightens.
Where the paper's t-test could be paired or unpaired, the unpaired test is used and the paired one
shown.

**What was found about the dandisets.**

- 000489 holds only the dishes of Fig. 2 and one sample image (Fig1F). The dishes of Figs. 3 to 6
  and the thermometry of Fig. 1E are not in either dandiset, so 13 of the 25 claims cannot be
  tested here. Dandisets 000548, 000549 and 000550 name this paper and match Figs. 3, 4 and 6 by
  title; the registry's list of the paper's datasets does not include them.
- The YO-PRO-1 channel of sub-Fig2-0-03 (unipolar) and sub-Fig2-25-04 (25%) is the same image,
  while their MemBrite channels differ: one recording deposited under two dishes.
- The files give no pixel size, electrode positions or polarity, or ROI; the pixel size of 0.65 um
  is supported by the electrode spacing (4.44 to 4.66 mm, Methods 4.5 mm). The image description
  names "Channel1" for both dyes; the data show index 0 is MemBrite and index 1 YO-PRO-1. It also
  speaks of 144 images (12 x 12) of a well of a six-well plate, where the paper describes 216
  images of 35 mm dishes.
- 000448 holds 9 cells, each as two files (forward and reverse trials) counted as 18 subjects. The
  reverse files store frames in acquisition order (descending delay), and their response is
  offset by 74 frames (1.85 us) from the forward trial of the same cell; neither is documented,
  nor the delay of each frame, the pole regions, or the field direction in the image. The first
  frame of most files is dark.